<a href="https://colab.research.google.com/github/ilincabaiasu/IB9AU/blob/main/Task_1_(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Required Task 1 (5)


*Note: GitHub does not support live Gradio outputs, please open Colab for running the code*

### Details

**Topic**: LLMs

**Sub-topic**: L3 - Embeddings Words to Sentences (L3_Embeddings_Words_To_Setences.ipynb)


---


**Instructions**:
Load the file financial_news.csv.

Last part of the sentence in each row of text contains an url. Remove this from text and create new column called URL and add the url.
Create sentence embeddings for the modified column text. Using Gradio, build a semantic search tool where the user enters some text (such as (“earnings surprise”, “regulatory fine”), and the top 5 closest records (based on cosine similarity) are output to the user.


---

**What I found interesting:**
- How easily you can create interactive interfaces using Gradio
- Shift into semantic search (model "understands" meaning)

In [ ]:
# imports
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr

In [ ]:
# upload financial_news.csv
df = pd.read_csv('financial_news.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Extract URLs from text
# Regex that matches a URL anywhere at the END of the string (after optional whitespace)
URL_PATTERN = re.compile(r'\s*(https?://\S+)\s*$')

def split_text_and_url(raw_text):
    """Return (clean_text, url) by removing the trailing URL from raw_text."""
    match = URL_PATTERN.search(str(raw_text))
    if match:
        url = match.group(1)
        clean = raw_text[:match.start()].strip()
        return clean, url
    return str(raw_text).strip(), None

# Apply to the 'text' column
df[['text', 'URL']] = df['text'].apply(
    lambda t: pd.Series(split_text_and_url(t))
)

print("Sample cleaned text:\n", df['text'].iloc[0])
print("\nSample URL:\n", df['URL'].iloc[0])
df[['text', 'URL']].head()

In [ ]:
# Generate sentence embeddings

# Load the same pre-trained sentence-transformer model used in the notebook
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all cleaned headlines — this produces a (N, 384) matrix
print("Generating sentence embeddings — this may take a moment...")
corpus_embeddings = sentence_model.encode(
    df['text'].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbedding matrix shape: {corpus_embeddings.shape}")
print(f"Each sentence is represented as a {corpus_embeddings.shape[1]}-dimensional vector")

Define semantic search function

Given a user query, we:
1. Embed the query with the same model
2. Compute cosine similarity against all corpus embeddings
3. Return the top 5 matches

In [ ]:
def semantic_search(query: str) -> pd.DataFrame:
    """
    Encode the query, compute cosine similarity with all corpus embeddings,
    and return the top 5 most similar records as a DataFrame.
    """
    if not query.strip():
        return pd.DataFrame({'Message': ['Please enter a search query.']})

    # Step 1: Embed the query (same model, same vector space)
    query_embedding = sentence_model.encode([query], convert_to_numpy=True)

    # Step 2: Cosine similarity — shape (1, N)
    similarities = cosine_similarity(query_embedding, corpus_embeddings)[0]

    # Step 3: Get indices of top 5 results (descending order)
    top5_indices = np.argsort(similarities)[::-1][:5]

    # Step 4: Build a results DataFrame
    results = df.iloc[top5_indices][['text', 'URL']].copy()
    results.insert(0, 'Similarity', similarities[top5_indices].round(4))
    results.reset_index(drop=True, inplace=True)
    results.index += 1  # Start ranking from 1
    results.index.name = 'Rank'

    return results

# Quick sanity check
test_results = semantic_search("earnings surprise")
print("Test query: 'earnings surprise'")
test_results

Building the Gradio Semantic Search Tool

We create an interactive interface where the user types a query (e.g. `"earnings surprise"` or `"regulatory fine"`) and receives the top 5 matching financial news headlines with their similarity scores and source URLs.

In [ ]:
# --- Gradio Interface ---

with gr.Blocks(title="Financial News Semantic Search") as demo:

    gr.Markdown("""
    # 📰 Financial News Semantic Search
    Enter a topic or phrase and retrieve the **top 5 most semantically similar**
    financial news headlines, ranked by cosine similarity.

    *Powered by `all-MiniLM-L6-v2` sentence embeddings*
    """)

    with gr.Row():
        query_box = gr.Textbox(
            label="Search Query",
            placeholder='e.g. "earnings surprise" or "regulatory fine"',
            lines=1,
            scale=4
        )
        search_btn = gr.Button("🔍 Search", variant="primary", scale=1)

    gr.Examples(
        examples=[
            ["earnings surprise"],
            ["regulatory fine"],
            ["merger acquisition deal"],
            ["stock market crash"],
            ["interest rate hike"],
            ["CEO resignation"],
        ],
        inputs=query_box,
        label="Example Queries"
    )

    results_table = gr.Dataframe(
        label="Top 5 Matching Headlines",
        headers=["Similarity", "Headline (text)", "URL"],
        wrap=True
    )

    # Wire up the search button and Enter key
    search_btn.click(fn=semantic_search, inputs=query_box, outputs=results_table)
    query_box.submit(fn=semantic_search, inputs=query_box, outputs=results_table)

    gr.Markdown("""
    ---
    **Similarity score** ranges from 0 (unrelated) to 1 (identical meaning).
    Scores above 0.4 typically indicate strong semantic relevance.
    """)

demo.launch()